In [0]:
# from datetime import date, timedelta

# # Remove this before running Data Factory Pipeline
# start_date = date.today() - timedelta(1)

# bronze_adls = "abfss://bronze@datalukerimas.dfs.core.windows.net/"
# silver_adls = "abfss://silver@datalukerimas.dfs.core.windows.net/"

import json

# Retrieve the bronze_params directly as a widget
bronze_params = dbutils.widgets.get("bronze_params")
print(f"Raw bronze_params: {bronze_params}")

output_data = json.loads(bronze_params)

# Access individual variables
start_date = output_data.get("start_date", "")
end_date = output_data.get("end_date", "")
bronze_adls = output_data.get("bronze_adls", "")
silver_adls = output_data.get("silver_adls", "")
gold_adls = output_data.get("gold_adls", "")

print(f"Start Date: {start_date}, Bronze ADLS: {bronze_adls}")


In [0]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta


In [0]:
# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"{bronze_adls}/{start_date}_earthquake_data.json")


In [0]:
df.head()

Row(geometry=Row(coordinates=[-151.258, 59.242, 0.1], type='Point'), id='aka2026nhkbzo', properties=Row(alert=None, cdi=None, code='a2026nhkbzo', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026nhkbzo&format=geojson', dmin=0.3, felt=None, gap=160, ids=',aka2026nhkbzo,', mag=2.3, magType='ml', mmi=None, net='ak', nst=37, place='33 km SE of Seldovia, Alaska', rms=0.8, sig=81, sources=',ak,', status='automatic', time=1783382378372, title='M 2.3 - 33 km SE of Seldovia, Alaska', tsunami=0, type='earthquake', types=',origin,phase-data,', tz=None, updated=1783382540726, url='https://earthquake.usgs.gov/earthquakes/eventpage/aka2026nhkbzo'), type='Feature')

In [0]:
# Reshape earthquake data
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)


In [0]:
df.head()


Row(id='aka2026nhkbzo', longitude=-151.258, latitude=59.242, elevation=0.1, title='M 2.3 - 33 km SE of Seldovia, Alaska', place_description='33 km SE of Seldovia, Alaska', sig=81, mag=2.3, magType='ml', time=1783382378372, updated=1783382540726)

In [0]:
from pyspark.sql.functions import (
    col, isnull, when, to_timestamp
)

# --- Null Handling ---
# Replace nulls in key fields with defaults or flags
df_silver = (
    df
    .withColumn("longitude", when(isnull(col("longitude")), 0.0).otherwise(col("longitude")))
    .withColumn("latitude", when(isnull(col("latitude")), 0.0).otherwise(col("latitude")))
    .withColumn("elevation", when(isnull(col("elevation")), 0.0).otherwise(col("elevation")))
    .withColumn("title", when(isnull(col("title")), "Unknown Title").otherwise(col("title")))
    .withColumn("place_description", when(isnull(col("place_description")), "Unknown Location").otherwise(col("place_description")))
    .withColumn("sig", when(isnull(col("sig")), 0).otherwise(col("sig")))
    .withColumn("mag", when(isnull(col("mag")), -1.0).otherwise(col("mag")))
    .withColumn("magType", when(isnull(col("magType")), "NA").otherwise(col("magType")))
)

# --- Timestamp Conversion ---
# Convert epoch milliseconds → proper Spark TimestampType
df_silver = (
    df_silver
    .withColumn("time", to_timestamp((col("time") / 1000).cast("double")))
    .withColumn("updated", to_timestamp((col("updated") / 1000).cast("double")))
)


In [0]:
df_silver.head()

Row(id='aka2026nhkbzo', longitude=-151.258, latitude=59.242, elevation=0.1, title='M 2.3 - 33 km SE of Seldovia, Alaska', place_description='33 km SE of Seldovia, Alaska', sig=81, mag=2.3, magType='ml', time=datetime.datetime(2026, 7, 6, 23, 59, 38, 372000), updated=datetime.datetime(2026, 7, 7, 0, 2, 20, 726000))

In [0]:
# Save the transformed DataFrame to the Silver container
silver_output_path = f"{silver_adls}earthquake_events_silver/"

# Append DataFrame to Silver container in Parquet format
df_silver.write.mode('append').parquet(silver_output_path)


In [0]:
# dbutils.fs.rm(silver_output_path, True)
dbutils.notebook.exit(silver_output_path)

True